In [16]:
import pandas as pd
import numpy as np

# 表 A：订单流水 (含重复录入的脏数据)
df_orders = pd.DataFrame({
    'Order_id': ['O1', 'O2', 'O3', 'O4', 'O5', 'O6', 'O2'], # O2 出现了两次，可能是系统录入错误
    'shop_id': ['S01', 'S01', 'S02', 'S02', 'S03', 'S01', 'S01'],
    'amount': [1000, 2000, 1500, 80, 120, 3000, 2000]
})

# 表 B：店铺信息 (含未注销的旧记录)
df_shops = pd.DataFrame({
    'Shop_Id ': ['S01', 'S01', 'S02', 'S03'],
    'shop_name': ['旗舰店', '旗舰店_旧', '社区店', '加盟店'], # S01 有两条记录，会引发合并爆炸
    'city': ['北京', '北京', '上海', '广州']
})

In [20]:
# uniform column names
df_orders.columns = [col.strip().lower() for col in df_orders.columns]
df_shops.columns = [col.strip().lower() for col in df_shops.columns]

# remove duplicates of df_shops
df_shops = df_shops.drop_duplicates(subset='shop_id',keep='first',ignore_index=True)

# remove duplicates of df_orders
df_orders = df_orders.drop_duplicates(subset='order_id',keep='first',ignore_index=True)

# merge df_orders and df_shops
df_merge = pd.merge(df_orders,df_shops,on='shop_id',how='left',indicator=True,validate='many_to_one')

# caculate the difference in the number of rows
diff = len(df_merge) - len(df_orders) 
print(f"数据增加了{diff}行")
print(df_merge)

# Calculate the total sales volume, average transaction value and total number of orders for each city.
# 针对不同列，下达不同的聚合密令
city_summary = df_merge.groupby('city').agg(
    total_sales=('amount', 'sum'),      # 总销售额
    avg_amount=('amount', 'mean'),     # 平均客单价
    order_count=('order_id', 'count')   # 订单总数
)

print("🏙️ 城市维度财务审计表：")
print(city_summary)
print(df_merge)

# 这里的 values 是 amount，我们要看的是每个城市在不同店铺类型的销售总额
final_pivot = df_merge.pivot_table(
    index='city', 
    columns='shop_name', 
    values='amount', 
    aggfunc='sum', 
    fill_value=0
)

print("\n📊 终极财务透视报表：")
print(final_pivot)

数据增加了0行
  order_id shop_id  amount shop_name city _merge
0       O1     S01    1000       旗舰店   北京   both
1       O2     S01    2000       旗舰店   北京   both
2       O3     S02    1500       社区店   上海   both
3       O4     S02      80       社区店   上海   both
4       O5     S03     120       加盟店   广州   both
5       O6     S01    3000       旗舰店   北京   both
🏙️ 城市维度财务审计表：
      total_sales  avg_amount  order_count
city                                      
上海           1580       790.0            2
北京           6000      2000.0            3
广州            120       120.0            1
  order_id shop_id  amount shop_name city _merge
0       O1     S01    1000       旗舰店   北京   both
1       O2     S01    2000       旗舰店   北京   both
2       O3     S02    1500       社区店   上海   both
3       O4     S02      80       社区店   上海   both
4       O5     S03     120       加盟店   广州   both
5       O6     S01    3000       旗舰店   北京   both

📊 终极财务透视报表：
shop_name  加盟店   旗舰店   社区店
city                      
上海        